# Notebook 01 - Detecção de Componentes

**Tech Challenge - Fase 5 - Hackaton FIAP**
**Modelagem de Ameaças utilizando IA (STRIDE)**

Este notebook executa os Passos 1 e 2 do pipeline:
- **Passo 1**: Identificar provedor cloud (AWS ou Azure) via Claude Vision
- **Passo 2a**: Detectar componentes visuais com Florence-2 (bounding boxes)
- **Passo 2b**: Classificar cada componente com Claude Vision (nome exato do serviço)
- **Passo 2c**: Analisar topologia do diagrama com Claude Vision (grupos, conexões, fluxo)

**GPU necessária**: T4 ou A100

## 1. Setup e Dependências

In [ ]:
# Instalar dependências
!pip install -q transformers anthropic supervision timm einops
# Nota: flash_attn (Flash Attention) não é utilizada — a compilação falha no Colab
# por incompatibilidade com CUDA. Florence-2 funciona normalmente sem ela,
# utilizando a implementação padrão de atenção do PyTorch.

In [ ]:
import os
import json
import re
import base64
import time
from pathlib import Path

import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import supervision as sv
from transformers import AutoProcessor, AutoModelForCausalLM
import anthropic

print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. Montar Google Drive e Configurar Caminhos

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

# Caminhos no Google Drive
DRIVE_BASE = Path('/content/drive/MyDrive/hackaton-stride')
IMAGES_DIR = DRIVE_BASE / 'test_images'
OUTPUT_DIR = DRIVE_BASE / 'outputs' / 'detections'

# Criar diretórios se não existirem
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Chave API do Claude (configurar em Secrets do Colab)
ANTHROPIC_API_KEY = userdata.get('ANTHROPIC_API_KEY')
client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print(f"Diretório de imagens: {IMAGES_DIR}")
print(f"Diretório de output: {OUTPUT_DIR}")
print(f"Imagens disponíveis: {list(IMAGES_DIR.glob('*.png'))}")

In [ ]:
# Selecionar imagem de teste
# Altere o índice para testar com outra imagem (1 = AWS, 2 = Azure)
ARCH_INDEX = 1
IMAGE_PATH = IMAGES_DIR / f'arquitetura {ARCH_INDEX}.png'

# Carregar e exibir a imagem
image = Image.open(IMAGE_PATH).convert('RGB')
print(f"Imagem: {IMAGE_PATH.name}")
print(f"Tamanho: {image.size}")

plt.figure(figsize=(14, 10))
plt.imshow(image)
plt.title(f'Arquitetura {ARCH_INDEX}')
plt.axis('off')
plt.show()

## 3. Passo 1 — Identificar Provedor Cloud (Claude Vision)

In [ ]:
def encode_image_base64(image_path: str) -> str:
    """Codifica imagem em base64 para enviar ao Claude Vision."""
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')

def encode_pil_image_base64(pil_image: Image.Image) -> str:
    """Codifica imagem PIL em base64."""
    import io
    buffer = io.BytesIO()
    pil_image.save(buffer, format='PNG')
    return base64.b64encode(buffer.getvalue()).decode('utf-8')

def clean_json_text(text: str) -> str:
    """
    Limpa texto retornado pelo Claude para extrair JSON válido.
    Remove code fences, trailing commas e outros artefatos.
    """
    text = re.sub(r'```(?:json)?\s*\n?', '', text).strip()
    text = re.sub(r',\s*([}\]])', r'\1', text)
    return text

def identify_cloud_provider(image_path: str) -> str:
    """
    Passo 1: Identifica o provedor cloud (AWS ou Azure) a partir da imagem.
    Usa Claude Vision para analisar logotipos, cores e nomenclatura.
    """
    image_b64 = encode_image_base64(image_path)

    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=100,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': image_b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': (
                            'Analise esta imagem de diagrama de arquitetura de software. '
                            'Identifique o provedor cloud principal baseando-se nos logotipos, '
                            'ícones, cores e nomes de serviços visíveis. '
                            'Responda APENAS com uma palavra: "AWS" ou "Azure".'
                        )
                    }
                ]
            }
        ]
    )

    provider = response.content[0].text.strip().upper()
    # Normalizar resposta
    if 'AWS' in provider:
        return 'AWS'
    elif 'AZURE' in provider:
        return 'Azure'
    else:
        print(f"Resposta inesperada: {provider}. Assumindo AWS.")
        return 'AWS'

In [ ]:
# Executar Passo 1
t0 = time.time()
provider = identify_cloud_provider(str(IMAGE_PATH))
t_provider = time.time() - t0

print(f"Provedor identificado: {provider}")
print(f"Tempo: {t_provider:.2f}s")

## 4. Passo 2a — Detecção de Componentes com Florence-2

In [ ]:
# Carregar modelo Florence-2 (versão fine-tuned para melhor detecção)
FLORENCE_MODEL_ID = 'microsoft/Florence-2-large-ft'

print("Carregando Florence-2-large-ft...")
florence_processor = AutoProcessor.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True
)
florence_model = AutoModelForCausalLM.from_pretrained(
    FLORENCE_MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.float16
).to('cuda')
print("Florence-2-large-ft carregado com sucesso!")

In [ ]:
def run_florence_task(image: Image.Image, task: str, text_input: str = None) -> dict:
    """Executa uma task do Florence-2 e retorna o resultado bruto."""
    prompt = task if text_input is None else task + text_input
    inputs = florence_processor(
        text=prompt,
        images=image,
        return_tensors='pt'
    ).to('cuda', torch.float16)

    with torch.no_grad():
        generated_ids = florence_model.generate(
            input_ids=inputs['input_ids'],
            pixel_values=inputs['pixel_values'],
            max_new_tokens=4096,
            num_beams=3
        )

    generated_text = florence_processor.batch_decode(
        generated_ids, skip_special_tokens=False
    )[0]

    result = florence_processor.post_process_generation(
        generated_text,
        task=task,
        image_size=(image.width, image.height)
    )
    return result


def get_grounding_phrases(provider: str) -> list:
    """Retorna frases de grounding específicas por provedor (1 serviço por query)."""
    # Termos genéricos visuais para diagramas de arquitetura
    phrases = [
        "icon", "logo", "symbol", "box", "label", "component", "service",
        "database", "server", "firewall", "load balancer", "storage", "gateway",
        "cloud", "network", "security", "monitoring", "backup", "cache",
        "user", "developer", "client", "actor",
        "arrow", "rectangle", "text", "diagram",
        "subnet", "zone", "region", "group",
    ]
    # Termos específicos por provedor (individuais para máxima cobertura)
    if provider == 'AWS':
        phrases.extend([
            "AWS WAF", "AWS Shield", "CloudFront", "ALB", "NLB",
            "VPC", "Subnet", "EC2", "Auto Scaling", "Auto Scaling Group",
            "RDS", "RDS Primary", "RDS Secondary", "Aurora",
            "ElastiCache", "EFS", "S3", "NFS",
            "CloudTrail", "CloudWatch", "KMS", "Key Management Service",
            "Backup", "AWS Backup",
            "SES", "Simple Email Service",
            "API Gateway", "Lambda", "Step Functions",
            "SQS", "SNS", "Cognito", "IAM",
            "Availability Zone", "Multi-AZ",
            "SEI", "SIP", "Solr",
            "Public Subnet", "Private Subnet",
            "Application Load Balancer",
            "Elastic File System",
        ])
    else:
        phrases.extend([
            "Azure Firewall", "Azure CDN", "Load Balancer",
            "VNet", "Subnet", "App Service", "VM", "VMSS",
            "Azure SQL", "Cosmos DB", "Azure Blob", "Key Vault",
            "Azure Monitor", "Azure Backup", "Service Bus",
            "API Management", "API Gateway", "Logic Apps",
            "Microsoft Entra", "Entra ID",
            "Developer Portal", "REST", "SOAP",
            "Resource Group", "HTTP", "HTTPS",
            "SaaS", "SaaS Service", "Web Service",
            "Azure Services", "Backend",
            "workflow", "authentication", "identity",
        ])
    return phrases


def run_all_detections(image: Image.Image, phrases: list, offset_x: int = 0, offset_y: int = 0) -> dict:
    """
    Executa todas as estratégias de detecção em uma imagem (ou tile).
    Ajusta coordenadas pelo offset para tiles.
    """
    boxes, labels = [], []

    def add_boxes(new_boxes, new_labels):
        for bbox in new_boxes:
            x1, y1, x2, y2 = bbox
            boxes.append([x1 + offset_x, y1 + offset_y,
                          x2 + offset_x, y2 + offset_y])
        labels.extend(new_labels)

    # OCR com regiões
    try:
        result = run_florence_task(image, '<OCR_WITH_REGION>')
        if '<OCR_WITH_REGION>' in result:
            data = result['<OCR_WITH_REGION>']
            for qbox, label in zip(data.get('quad_boxes', []), data.get('labels', [])):
                if len(qbox) == 8:
                    x_coords = [qbox[i] for i in range(0, 8, 2)]
                    y_coords = [qbox[i] for i in range(1, 8, 2)]
                    boxes.append([min(x_coords) + offset_x, min(y_coords) + offset_y,
                                  max(x_coords) + offset_x, max(y_coords) + offset_y])
                    labels.append(label.strip())
    except Exception:
        pass

    # Phrase grounding (1 serviço por query)
    for phrase in phrases:
        try:
            result = run_florence_task(image, '<CAPTION_TO_PHRASE_GROUNDING>', phrase)
            if '<CAPTION_TO_PHRASE_GROUNDING>' in result:
                data = result['<CAPTION_TO_PHRASE_GROUNDING>']
                b = data.get('bboxes', [])
                l = data.get('labels', [])
                add_boxes(b, l)
        except Exception:
            pass

    # OD + Dense Region Caption + Region Proposal
    for task in ['<OD>', '<DENSE_REGION_CAPTION>', '<REGION_PROPOSAL>']:
        try:
            result = run_florence_task(image, task)
            if task in result:
                data = result[task]
                b = data.get('bboxes', [])
                l = data.get('labels', ['region'] * len(b))
                add_boxes(b, l)
        except Exception:
            pass

    return {'bboxes': boxes, 'labels': labels}


def detect_components_florence(image: Image.Image, provider: str) -> dict:
    """
    Passo 2a: Detecta regiões de interesse usando múltiplas estratégias Florence-2.

    Estratégias:
    1. OCR_WITH_REGION: detecta textos (nomes dos serviços)
    2. CAPTION_TO_PHRASE_GROUNDING: 1 serviço por query para máxima cobertura
    3. OD + DENSE_REGION_CAPTION + REGION_PROPOSAL: detecção geral
    4. Tiling 3x3 com 20% overlap: repete todas as estratégias em cada tile
    """
    phrases = get_grounding_phrases(provider)

    # --- Imagem completa ---
    print(f"  [1/2] Imagem completa ({len(phrases)} phrases de grounding)...")
    full_result = run_all_detections(image, phrases)
    print(f"    -> {len(full_result['bboxes'])} regiões")

    all_boxes = list(full_result['bboxes'])
    all_labels = list(full_result['labels'])

    # --- Tiling 3x3 com overlap ---
    print("  [2/2] Tiling 3x3 com 20% overlap...")
    w, h = image.size
    overlap = 0.2
    cols, rows = 3, 3
    tile_w = int(w / cols * (1 + overlap))
    tile_h = int(h / rows * (1 + overlap))
    step_x = (w - tile_w) / (cols - 1) if cols > 1 else 0
    step_y = (h - tile_h) / (rows - 1) if rows > 1 else 0

    tile_count = 0
    for row in range(rows):
        for col in range(cols):
            tx1 = int(col * step_x)
            ty1 = int(row * step_y)
            tx2 = min(tx1 + tile_w, w)
            ty2 = min(ty1 + tile_h, h)

            tile = image.crop((tx1, ty1, tx2, ty2))
            tile_result = run_all_detections(tile, phrases, offset_x=tx1, offset_y=ty1)
            all_boxes.extend(tile_result['bboxes'])
            all_labels.extend(tile_result['labels'])
            tile_count += len(tile_result['bboxes'])
            print(f"    Tile ({row},{col}): {len(tile_result['bboxes'])} regiões")

    print(f"    -> {tile_count} regiões por tiling")
    print(f"  Total bruto: {len(all_boxes)} regiões")
    return {'bboxes': all_boxes, 'labels': all_labels}


def filter_and_merge_boxes(detections: dict, image_size: tuple,
                           min_area_ratio: float = 0.0005,
                           max_area_ratio: float = 0.05,
                           iou_threshold: float = 0.5) -> dict:
    """
    Filtra bounding boxes e remove duplicatas via NMS.
    max_area_ratio=0.05 elimina detecções gigantes (>5% da imagem)
    que são artefatos de frases genéricas como 'box', 'icon'.
    """
    img_w, img_h = image_size
    img_area = img_w * img_h

    boxes = []
    labels = []

    for bbox, label in zip(detections['bboxes'], detections['labels']):
        x1, y1, x2, y2 = bbox
        # Clamp ao tamanho da imagem
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(img_w, x2), min(img_h, y2)
        w = x2 - x1
        h = y2 - y1
        area = w * h
        ratio = area / img_area

        if min_area_ratio <= ratio <= max_area_ratio and w > 5 and h > 5:
            boxes.append([x1, y1, x2, y2])
            labels.append(label)

    if not boxes:
        return {'bboxes': [], 'labels': []}

    # NMS para remover duplicatas
    boxes_np = np.array(boxes)
    keep = []
    used = set()

    for i in range(len(boxes_np)):
        if i in used:
            continue
        keep.append(i)
        for j in range(i + 1, len(boxes_np)):
            if j in used:
                continue
            xi1 = max(boxes_np[i][0], boxes_np[j][0])
            yi1 = max(boxes_np[i][1], boxes_np[j][1])
            xi2 = min(boxes_np[i][2], boxes_np[j][2])
            yi2 = min(boxes_np[i][3], boxes_np[j][3])
            inter = max(0, xi2 - xi1) * max(0, yi2 - yi1)
            area_i = (boxes_np[i][2] - boxes_np[i][0]) * (boxes_np[i][3] - boxes_np[i][1])
            area_j = (boxes_np[j][2] - boxes_np[j][0]) * (boxes_np[j][3] - boxes_np[j][1])
            iou = inter / (area_i + area_j - inter + 1e-6)
            if iou > iou_threshold:
                used.add(j)

    filtered_boxes = [boxes[i] for i in keep]
    filtered_labels = [labels[i] for i in keep]

    return {'bboxes': filtered_boxes, 'labels': filtered_labels}


def deduplicate_components(components: list) -> list:
    """
    Remove componentes duplicados após classificação Claude.
    Agrupa por nome e, para cada nome, mantém apenas instâncias
    espacialmente distintas (centros distantes > 100px).
    De cada cluster espacial, mantém o bbox de menor área (mais preciso).
    """
    from collections import defaultdict
    import math

    if not components:
        return components

    # Agrupar por nome
    by_name = defaultdict(list)
    for comp in components:
        by_name[comp['name']].append(comp)

    result = []
    for name, group in by_name.items():
        if len(group) == 1:
            result.append(group[0])
            continue

        # Clusterizar por proximidade espacial
        # bbox é [x, y, w, h] — calcular centro
        centers = []
        for c in group:
            cx = c['bbox'][0] + c['bbox'][2] / 2
            cy = c['bbox'][1] + c['bbox'][3] / 2
            area = c['bbox'][2] * c['bbox'][3]
            centers.append((cx, cy, area, c))

        clusters = []
        used = set()
        for i, (cx1, cy1, a1, c1) in enumerate(centers):
            if i in used:
                continue
            cluster = [(a1, c1)]
            used.add(i)
            for j, (cx2, cy2, a2, c2) in enumerate(centers):
                if j in used:
                    continue
                dist = math.sqrt((cx1 - cx2)**2 + (cy1 - cy2)**2)
                if dist < 100:  # mesmo componente se centros < 100px
                    cluster.append((a2, c2))
                    used.add(j)
            # Do cluster, manter o de menor área (mais preciso)
            cluster.sort(key=lambda x: x[0])
            result.append(cluster[0][1])

    # Reordenar por posição (top-left para bottom-right)
    result.sort(key=lambda c: (c['bbox'][1], c['bbox'][0]))

    # Re-numerar IDs
    for i, comp in enumerate(result):
        comp['id'] = i

    return result

In [ ]:
# Executar Passo 2a: Detecção com Florence-2
print("Detectando componentes com Florence-2 (múltiplas estratégias)...\n")

t0 = time.time()
raw_detections = detect_components_florence(image, provider)
t_florence = time.time() - t0

print(f"\nDetecções brutas: {len(raw_detections['bboxes'])} regiões")
print(f"Tempo Florence-2: {t_florence:.2f}s")

# Filtrar e remover duplicatas
detections = filter_and_merge_boxes(raw_detections, image.size)
print(f"Detecções após filtro + NMS: {len(detections['bboxes'])} regiões")

In [ ]:
# Visualizar detecções brutas do Florence-2
fig, ax = plt.subplots(1, 1, figsize=(16, 12))
ax.imshow(image)

colors = plt.cm.Set3(np.linspace(0, 1, len(detections['bboxes'])))

for idx, (bbox, label) in enumerate(zip(detections['bboxes'], detections['labels'])):
    x1, y1, x2, y2 = bbox
    w = x2 - x1
    h = y2 - y1
    rect = patches.Rectangle(
        (x1, y1), w, h,
        linewidth=2, edgecolor=colors[idx], facecolor='none'
    )
    ax.add_patch(rect)
    ax.text(x1, y1 - 5, f'{idx}: {label}', fontsize=7,
            color='white', backgroundcolor=colors[idx])

ax.set_title(f'Florence-2: {len(detections["bboxes"])} regiões detectadas')
ax.axis('off')
plt.tight_layout()
plt.show()

## 5. Passo 2b — Classificação de Componentes com Claude Vision

In [ ]:
# Classes de detecção do projeto
COMPONENT_CLASSES = {
    'waf_firewall': ['AWS WAF', 'AWS Shield', 'Azure Firewall'],
    'cdn': ['Amazon CloudFront', 'Azure CDN'],
    'load_balancer': ['ALB', 'NLB', 'Azure Load Balancer'],
    'vpc_vnet': ['VPC', 'VNet', 'Subnet'],
    'compute': ['EC2', 'SEI/SIP', 'App Service', 'VM'],
    'auto_scaling': ['Auto Scaling Group', 'VMSS'],
    'orchestrator': ['Logic Apps', 'Step Functions', 'Lambda'],
    'database': ['RDS', 'Aurora', 'Azure SQL', 'Cosmos DB'],
    'cache': ['ElastiCache', 'Azure Cache for Redis'],
    'storage': ['S3', 'EFS', 'Azure Blob', 'NFS'],
    'api_gateway': ['API Gateway', 'Azure API Management'],
    'developer_portal': ['Developer Portal'],
    'web_service': ['REST', 'SOAP', 'SaaS Service'],
    'auth_identity': ['IAM', 'Microsoft Entra', 'Cognito'],
    'kms_encryption': ['AWS KMS', 'Azure Key Vault'],
    'monitoring': ['CloudWatch', 'CloudTrail', 'Azure Monitor'],
    'backup': ['AWS Backup', 'Azure Backup'],
    'messaging': ['SES', 'SQS', 'SNS', 'Azure Service Bus'],
    'user_actor': ['Usuário', 'Developer', 'Client'],
}

# Lista plana de todos os serviços para o prompt
ALL_SERVICES = []
for cls, services in COMPONENT_CLASSES.items():
    for svc in services:
        ALL_SERVICES.append(f'{svc} ({cls})')

SERVICES_LIST = ', '.join(ALL_SERVICES)

In [ ]:
def classify_component(crop_image: Image.Image, provider: str,
                       full_image_b64: str) -> dict:
    """
    Passo 2b: Classifica um crop de componente usando Claude Vision.
    Envia o crop + imagem completa para contexto.

    Retorna dict com 'name' (nome do serviço) e 'class' (classe genérica).
    """
    crop_b64 = encode_pil_image_base64(crop_image)

    prompt = f"""Analise este recorte de um diagrama de arquitetura {provider}.

A segunda imagem mostra o diagrama completo para contexto.

Identifique o componente/serviço cloud mostrado no recorte.

Serviços possíveis: {SERVICES_LIST}

IMPORTANTE: Responda APENAS com JSON puro, sem code fences, sem comentários, sem texto antes ou depois.
{{
  "name": "Nome exato do serviço (ex: Amazon RDS, AWS WAF)",
  "class": "classe genérica (ex: database, waf_firewall)"
}}

Se o recorte não contiver um componente cloud identificável, responda:
{{"name": "unknown", "class": "unknown"}}"""

    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=200,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': crop_b64
                        }
                    },
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': full_image_b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': prompt
                    }
                ]
            }
        ]
    )

    text = response.content[0].text.strip()
    # Limpar e extrair JSON
    try:
        cleaned = clean_json_text(text)
        start = cleaned.find('{')
        end = cleaned.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end])
    except json.JSONDecodeError:
        pass

    return {'name': 'unknown', 'class': 'unknown'}

In [ ]:
# Executar Passo 2b: Classificar cada componente
print(f"Classificando {len(detections['bboxes'])} componentes com Claude Vision...")
print(f"Provedor: {provider}\n")

full_image_b64 = encode_image_base64(str(IMAGE_PATH))
components = []

t0 = time.time()
for idx, bbox in enumerate(detections['bboxes']):
    x1, y1, x2, y2 = [int(c) for c in bbox]

    # Recortar região com margem
    margin = 10
    crop_x1 = max(0, x1 - margin)
    crop_y1 = max(0, y1 - margin)
    crop_x2 = min(image.width, x2 + margin)
    crop_y2 = min(image.height, y2 + margin)
    crop = image.crop((crop_x1, crop_y1, crop_x2, crop_y2))

    # Classificar com Claude Vision
    result = classify_component(crop, provider, full_image_b64)

    component = {
        'id': idx,
        'name': result.get('name', 'unknown'),
        'class': result.get('class', 'unknown'),
        'bbox': [x1, y1, x2 - x1, y2 - y1],  # [x, y, w, h]
        'provider': provider,
        'florence_label': detections['labels'][idx]
    }
    components.append(component)

    status = f"  [{idx+1}/{len(detections['bboxes'])}] {result.get('name', '?')} ({result.get('class', '?')})"
    print(status)

    # Rate limiting
    time.sleep(0.5)

t_classify = time.time() - t0

# Filtrar componentes desconhecidos
known_components = [c for c in components if c['class'] != 'unknown']
print(f"\nComponentes identificados: {len(known_components)}/{len(components)}")

# Deduplicar: agrupar por nome + proximidade espacial
before_dedup = len(known_components)
known_components = deduplicate_components(known_components)
print(f"Após deduplicação espacial: {len(known_components)} (removidos {before_dedup - len(known_components)} duplicatas)")
print(f"Tempo classificação: {t_classify:.2f}s")

## 5.1 Passo 2c — Análise de Topologia com Claude Vision

Analisa a imagem completa para identificar:
- **Grupos/containers**: VPC, Subnets, Availability Zones, Resource Groups
- **Conexões**: setas e linhas entre componentes
- **Fluxo de dados**: ordem do tráfego de ponta a ponta

In [ ]:
def analyze_topology(image_path: str, components: list, provider: str) -> dict:
    """
    Passo 2c: Analisa a topologia do diagrama usando Claude Vision.
    Identifica grupos (containers), conexões entre componentes e fluxo de dados.
    """
    image_b64 = encode_image_base64(image_path)

    # Lista de componentes detectados para contextualizar
    comp_list = "\n".join(
        f"  - {c['name']} ({c['class']})"
        for c in components
    )

    prompt = f"""Analise este diagrama de arquitetura {provider}.

Os seguintes componentes foram detectados:
{comp_list}

Identifique a topologia do diagrama analisando visualmente:

1. **Grupos/containers**: caixas pontilhadas ou sólidas que agrupam componentes (VPC, Subnets, Availability Zones, Resource Groups, Regions, etc.)
2. **Conexões**: setas ou linhas que conectam componentes entre si (indicando comunicação, fluxo de dados, dependência)
3. **Fluxo de dados**: a ordem do tráfego de ponta a ponta (do usuário até o backend)

IMPORTANTE: Responda APENAS com JSON puro, sem code fences (```), sem comentários, sem texto antes ou depois.
{{
  "groups": [
    {{
      "name": "Nome do grupo (ex: VPC, AZ-A Public Subnet)",
      "type": "Tipo (ex: vpc, subnet, availability_zone, region, resource_group)",
      "contains": ["Nome do componente 1", "Nome do componente 2"],
      "parent": "Nome do grupo pai ou null se for raiz"
    }}
  ],
  "connections": [
    {{
      "from": "Nome do componente origem",
      "to": "Nome do componente destino",
      "description": "Breve descrição (ex: HTTPS, tráfego interno, replicação)"
    }}
  ],
  "data_flow": ["Componente 1", "Componente 2", "Componente 3"]
}}

Use EXATAMENTE os nomes dos componentes listados acima. Para grupos que contêm outros grupos, use a hierarquia pai/filho via "parent"."""

    response = client.messages.create(
        model='claude-sonnet-4-6',
        max_tokens=4000,
        messages=[
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type': 'base64',
                            'media_type': 'image/png',
                            'data': image_b64
                        }
                    },
                    {
                        'type': 'text',
                        'text': prompt
                    }
                ]
            }
        ]
    )

    text = response.content[0].text.strip()
    try:
        cleaned = clean_json_text(text)
        start = cleaned.find('{')
        end = cleaned.rfind('}') + 1
        if start >= 0 and end > start:
            return json.loads(cleaned[start:end])
    except json.JSONDecodeError as e:
        print(f"  Erro ao parsear topologia: {e}")
        # Tentativa extra
        try:
            raw = text[text.find('{'):text.rfind('}') + 1]
            raw = re.sub(r',\s*([}\]])', r'\1', raw)
            return json.loads(raw)
        except (json.JSONDecodeError, ValueError):
            print(f"  Falha na segunda tentativa de parsing.")

    return {'groups': [], 'connections': [], 'data_flow': []}

In [ ]:
# Executar Passo 2c: Análise de Topologia
print("Analisando topologia do diagrama com Claude Vision...\n")

t0 = time.time()
topology = analyze_topology(str(IMAGE_PATH), known_components, provider)
t_topology = time.time() - t0

print(f"Grupos identificados: {len(topology.get('groups', []))}")
for g in topology.get('groups', []):
    parent = f" (dentro de {g['parent']})" if g.get('parent') else ""
    print(f"  - {g['name']} [{g.get('type', '?')}]{parent}: {len(g.get('contains', []))} componentes")

print(f"\nConexões identificadas: {len(topology.get('connections', []))}")
for conn in topology.get('connections', []):
    print(f"  - {conn['from']} -> {conn['to']}: {conn.get('description', '')}")

print(f"\nFluxo de dados: {' -> '.join(topology.get('data_flow', []))}")
print(f"\nTempo topologia: {t_topology:.2f}s")

## 6. Salvar Resultados

In [ ]:
# Montar JSON de saída
output_data = {
    'image': IMAGE_PATH.name,
    'provider': provider,
    'image_size': {'width': image.width, 'height': image.height},
    'total_detections': len(detections['bboxes']),
    'components': known_components,
    'topology': topology,
    'timing': {
        'provider_identification_s': round(t_provider, 2),
        'florence_detection_s': round(t_florence, 2),
        'claude_classification_s': round(t_classify, 2),
        'topology_analysis_s': round(t_topology, 2),
        'total_s': round(t_provider + t_florence + t_classify + t_topology, 2)
    }
}

# Salvar JSON
output_file = OUTPUT_DIR / f'componentes_arquitetura_{ARCH_INDEX}.json'
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(output_data, f, ensure_ascii=False, indent=2)

print(f"Resultados salvos em: {output_file}")
print(f"\nResumo:")
print(f"  Provedor: {provider}")
print(f"  Componentes detectados: {len(known_components)}")
print(f"  Grupos topológicos: {len(topology.get('groups', []))}")
print(f"  Conexões mapeadas: {len(topology.get('connections', []))}")
print(f"  Tempo total: {output_data['timing']['total_s']}s")
print(f"\nComponentes:")
for c in known_components:
    print(f"  - {c['name']} ({c['class']})")

## 7. Visualização com Supervision

In [ ]:
# Visualizar componentes detectados com bounding boxes
if known_components:
    # Preparar dados para supervision
    xyxy = np.array([
        [c['bbox'][0], c['bbox'][1],
         c['bbox'][0] + c['bbox'][2], c['bbox'][1] + c['bbox'][3]]
        for c in known_components
    ])

    detections_sv = sv.Detections(
        xyxy=xyxy,
        class_id=np.arange(len(known_components))
    )

    labels = [
        f"{c['name']}\n({c['class']})"
        for c in known_components
    ]

    # Anotar imagem
    img_np = np.array(image)

    box_annotator = sv.BoxAnnotator(
        thickness=2
    )
    label_annotator = sv.LabelAnnotator(
        text_scale=0.4,
        text_padding=4
    )

    annotated = box_annotator.annotate(
        scene=img_np.copy(),
        detections=detections_sv
    )
    annotated = label_annotator.annotate(
        scene=annotated,
        detections=detections_sv,
        labels=labels
    )

    # Exibir
    plt.figure(figsize=(18, 14))
    plt.imshow(annotated)
    plt.title(f'Arquitetura {ARCH_INDEX} ({provider}) - {len(known_components)} componentes detectados')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    # Salvar imagem anotada
    annotated_path = OUTPUT_DIR / f'annotated_arquitetura_{ARCH_INDEX}.png'
    Image.fromarray(annotated).save(annotated_path)
    print(f"Imagem anotada salva em: {annotated_path}")
else:
    print("Nenhum componente identificado para visualizar.")

## 8. Resumo da Execução

In [ ]:
# Tabela resumo
print("=" * 60)
print(f"RESUMO - Detecção de Componentes")
print("=" * 60)
print(f"Imagem: {IMAGE_PATH.name}")
print(f"Provedor: {provider}")
print(f"Tamanho: {image.size}")
print(f"")
print(f"Detecções Florence-2: {len(detections['bboxes'])}")
print(f"Componentes classificados: {len(known_components)}")
print(f"Grupos topológicos: {len(topology.get('groups', []))}")
print(f"Conexões mapeadas: {len(topology.get('connections', []))}")
print(f"")
print(f"Tempo - Identificação provedor: {t_provider:.2f}s")
print(f"Tempo - Florence-2: {t_florence:.2f}s")
print(f"Tempo - Classificação Claude: {t_classify:.2f}s")
print(f"Tempo - Topologia Claude: {t_topology:.2f}s")
print(f"Tempo total: {t_provider + t_florence + t_classify + t_topology:.2f}s")
print(f"")
print(f"Output: {output_file}")
print("=" * 60)

# Classes detectadas
from collections import Counter
class_counts = Counter(c['class'] for c in known_components)
print(f"\nClasses detectadas:")
for cls, count in class_counts.most_common():
    print(f"  {cls}: {count}")

## 9. Estimativa de Custo

In [ ]:
# Estimativa de custo do Notebook 01
# Preços Claude Sonnet 4.6: $3.00/1M tokens input, $15.00/1M tokens output
# Google Colab Pro: ~$10/mês (~0.014/hora considerando uso médio de 720h/mês)
# Cotação: US$ 1.00 = R$ 5.50

PRICE_INPUT = 3.00 / 1_000_000   # USD por token de input
PRICE_OUTPUT = 15.00 / 1_000_000  # USD por token de output
COLAB_HOUR = 0.014                # USD por hora (estimativa Colab Pro)
USD_BRL = 5.50

n_components = len(detections['bboxes'])  # regiões enviadas ao Claude (inclui unknowns)
total_time_h = (t_provider + t_florence + t_classify + t_topology) / 3600

# Passo 1: Identificação do provedor (1 chamada com imagem)
step1_input = 1_800   # ~1.8k tokens (imagem + prompt)
step1_output = 50
step1_cost = step1_input * PRICE_INPUT + step1_output * PRICE_OUTPUT

# Passo 2b: Classificação de componentes (N chamadas com 2 imagens cada)
step2b_input = n_components * 1_200   # ~1.2k tokens por chamada (2 imagens + prompt)
step2b_output = n_components * 80     # ~80 tokens de output por chamada
step2b_cost = step2b_input * PRICE_INPUT + step2b_output * PRICE_OUTPUT

# Passo 2c: Topologia (1 chamada com imagem + lista de componentes)
step2c_input = 2_500   # ~2.5k tokens (imagem + prompt com lista)
step2c_output = 2_000  # ~2k tokens de output (JSON com grupos/conexões)
step2c_cost = step2c_input * PRICE_INPUT + step2c_output * PRICE_OUTPUT

# Florence-2: roda localmente na GPU (sem custo API)
colab_cost = total_time_h * COLAB_HOUR

# Totais
claude_cost = step1_cost + step2b_cost + step2c_cost
total_cost = claude_cost + colab_cost

print("=" * 60)
print(f"ESTIMATIVA DE CUSTO - Notebook 01")
print("=" * 60)
print(f"")
print(f"Claude API (Sonnet 4.6):")
print(f"  Passo 1 - Provedor:      ${step1_cost:.4f} (R$ {step1_cost * USD_BRL:.4f})")
print(f"  Passo 2b - Classificação: ${step2b_cost:.4f} (R$ {step2b_cost * USD_BRL:.4f})")
print(f"    ({n_components} chamadas)")
print(f"  Passo 2c - Topologia:     ${step2c_cost:.4f} (R$ {step2c_cost * USD_BRL:.4f})")
print(f"  Subtotal Claude API:      ${claude_cost:.4f} (R$ {claude_cost * USD_BRL:.4f})")
print(f"")
print(f"Google Colab Pro (GPU):")
print(f"  Tempo de execução:        {total_time_h * 60:.1f} minutos")
print(f"  Subtotal Colab:           ${colab_cost:.4f} (R$ {colab_cost * USD_BRL:.4f})")
print(f"")
print(f"Florence-2: roda localmente na GPU (sem custo API)")
print(f"")
print(f"{'─' * 60}")
print(f"TOTAL ESTIMADO:             ${total_cost:.4f} (R$ {total_cost * USD_BRL:.4f})")
print(f"{'─' * 60}")
print(f"")
print(f"Nota: Valores aproximados. Tokens de imagem variam conforme")
print(f"resolução. Cotação: US$ 1.00 = R$ {USD_BRL:.2f}")